# 01 - Exploratory Data Analysis (EDA)
First look at the Snappfood reviews: how big the data is, whether the labels are balanced,
how long the comments are, and which words show up most.

## 0. Setup
Connect Google Drive, where the CSV file is stored.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DATA_PATH = "/content/drive/MyDrive/snappfood-sentiment/Snappfood - Sentiment Analysis.csv"

if os.path.exists(DATA_PATH):
    print("Data file found.")
else:
    print("Data file NOT found. Check the folder and file name in your Google Drive.")

In [ ]:
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

## 1. Load the raw data
The file is tab-separated. A few rows are broken, so we skip them.

In [ ]:
raw = pd.read_csv(DATA_PATH, sep="\t", on_bad_lines="skip", engine="python")

print("shape:", raw.shape)
print("columns:", raw.columns.tolist())
raw.head()

In [ ]:
# which values appear in the label column?
raw["label"].value_counts(dropna=False).head(10)

## 2. Minimal cleaning
- keep only rows labeled HAPPY or SAD
- drop empty comments
- drop comments that appear with **both** labels (we can't trust them)
- drop exact duplicates

Note: in the original file HAPPY = 0 and SAD = 1. We flip it so **positive = 1**, which is easier to read later.

In [ ]:
LABEL_MAP = {"HAPPY": 1, "SAD": 0}

def clean_data(df):
    n0 = len(df)
    df = df[["comment", "label"]].copy()

    df["label"] = df["label"].astype(str).str.strip().str.upper()
    df = df[df["label"].isin(LABEL_MAP)]
    n1 = len(df)

    df = df.dropna(subset=["comment"])
    df["comment"] = df["comment"].astype(str).str.strip()
    df = df[df["comment"] != ""]
    n2 = len(df)

    conflict = df.groupby("comment")["label"].nunique()
    conflict = conflict[conflict > 1].index
    df = df[~df["comment"].isin(conflict)]
    n3 = len(df)

    df = df.drop_duplicates(subset="comment")
    df["sentiment"] = df["label"].map(LABEL_MAP)
    df = df.reset_index(drop=True)

    print(f"raw rows:                   {n0:,}")
    print(f"invalid label removed:      {n0 - n1:,}")
    print(f"empty comment removed:      {n1 - n2:,}")
    print(f"conflicting labels removed: {n2 - n3:,}")
    print(f"duplicates removed:         {n3 - len(df):,}")
    print(f"final rows:                 {len(df):,}")
    return df

df = clean_data(raw)
df.head()

## 3. Label balance

In [ ]:
counts = df["label"].value_counts()
print(counts)
print((counts / len(df)).round(3))

ax = counts.plot(kind="bar", color=["#4C9A6A", "#C0504D"], rot=0)
ax.set_title("Label distribution")
ax.set_ylabel("count")
plt.show()

## 4. Comment length

In [ ]:
df["n_words"] = df["comment"].str.split().str.len()
df["n_chars"] = df["comment"].str.len()

df.groupby("label")[["n_words", "n_chars"]].describe().T

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=df[df["n_words"] <= 100], x="n_words", hue="label",
             bins=50, stat="density", common_norm=False)
plt.title("Words per comment (clipped at 100)")
plt.show()

In [ ]:
# a quick look at the shortest and longest comments
print(df.nsmallest(5, "n_words")[["comment", "label"]])
print()
print(df.nlargest(3, "n_words")[["comment", "label"]])

## 5. Most frequent words per label
No stopword removal yet, so words like «و» and «از» will be at the top.
That's exactly why we need preprocessing in the next notebook.

In [ ]:
def simple_tokens(text):
    return re.findall(r"[\u0600-\u06FF\u200c]+", text)

top = {}
for lab in ["HAPPY", "SAD"]:
    c = Counter(t for txt in df.loc[df["label"] == lab, "comment"] for t in simple_tokens(txt))
    top[lab] = pd.Series([w for w, _ in c.most_common(25)])

pd.concat(top, axis=1)

## 6. Key findings
*(fill in after running)*
- 
- 
- 